In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load CSVs into memory (keep full tables for later steps)
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')
log_df = pd.read_csv('log.csv')
error_df = pd.read_csv('error_logs.csv')

# Convert timestamps to timezone-aware datetimes in UTC+8 (Asia/Shanghai)
for df in (metric_df, trace_df, log_df, error_df):
    if 'timestamp' in df.columns:
        df['dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) metric.csv summary
metric_rows = len(metric_df)
metric_min = metric_df['dt'].min() if metric_rows > 0 else pd.NaT
metric_max = metric_df['dt'].max() if metric_rows > 0 else pd.NaT
metric_unique_cmdb = pd.Series(metric_df['cmdb_id'].dropna().unique()[:50])  # up to 50 sample
metric_kpi_counts = metric_df['kpi_name'].value_counts().reset_index()
metric_kpi_counts.columns = ['kpi_name', 'count']
metric_kpi_top20 = metric_kpi_counts.head(20)

metric_summary = pd.Series({
    'rows': metric_rows,
    'min_timestamp': metric_min,
    'max_timestamp': metric_max,
    'unique_cmdb_sample_count': metric_unique_cmdb.shape[0],
    'unique_cmdb_sample': metric_unique_cmdb.tolist()
})

# 2) trace.csv summary
trace_rows = len(trace_df)
trace_min = trace_df['dt'].min() if trace_rows > 0 else pd.NaT
trace_max = trace_df['dt'].max() if trace_rows > 0 else pd.NaT
trace_unique_cmdb = pd.Series(trace_df['cmdb_id'].dropna().unique()[:50])
trace_name_counts = trace_df['trace_name'].value_counts().reset_index()
trace_name_counts.columns = ['trace_name', 'count']
trace_name_top20 = trace_name_counts.head(20)

trace_summary = pd.Series({
    'rows': trace_rows,
    'min_timestamp': trace_min,
    'max_timestamp': trace_max,
    'unique_cmdb_sample_count': trace_unique_cmdb.shape[0],
    'unique_cmdb_sample': trace_unique_cmdb.tolist()
})

# 3) log.csv summary
log_rows = len(log_df)
log_min = log_df['dt'].min() if log_rows > 0 else pd.NaT
log_max = log_df['dt'].max() if log_rows > 0 else pd.NaT
log_unique_cmdb = pd.Series(log_df['cmdb_id'].dropna().unique()[:50])
log_name_counts = log_df['log_name'].value_counts().reset_index()
log_name_counts.columns = ['log_name', 'count']
log_name_top20 = log_name_counts.head(20)

log_summary = pd.Series({
    'rows': log_rows,
    'min_timestamp': log_min,
    'max_timestamp': log_max,
    'unique_cmdb_sample_count': log_unique_cmdb.shape[0],
    'unique_cmdb_sample': log_unique_cmdb.tolist()
})

# 4) error_logs.csv summary and top 20 rows (timestamp, cmdb_id, first 200 chars of message)
error_rows = len(error_df)
error_min = error_df['dt'].min() if error_rows > 0 else pd.NaT
error_max = error_df['dt'].max() if error_rows > 0 else pd.NaT
error_unique_cmdb = pd.Series(error_df['cmdb_id'].dropna().unique()[:50])

# ensure message column exists and create snippet column
if 'message' in error_df.columns:
    error_df['message_snip'] = error_df['message'].astype(str).str.slice(0, 200)
else:
    error_df['message_snip'] = ''

# select top 20 rows by timestamp ascending (compact)
error_top20 = error_df.sort_values('dt').loc[:, ['dt', 'cmdb_id', 'message_snip']].head(20)

error_summary = pd.Series({
    'rows': error_rows,
    'min_timestamp': error_min,
    'max_timestamp': error_max,
    'unique_cmdb_sample_count': error_unique_cmdb.shape[0],
    'unique_cmdb_sample': error_unique_cmdb.tolist()
})

# Final compact outputs (kept small as requested)
metric_summary, metric_kpi_top20, trace_summary, trace_name_top20, log_summary, log_name_top20, error_summary, error_top20

```
Out[1]:
```
summary = (
    "Summary of loaded telemetry (timestamps shown in UTC+8):\n"
    "- metric.csv: 319,703 rows; time range 2022-03-20 09:00 → 2022-03-20 10:00; 50 unique cmdb_id samples. "
    "Top KPI examples (by count): several container.node-6 network KPIs (each 2196), and multiple node-6 memory/fs/task KPIs (~2074). "
    "See variable metric_kpi_top20 for the full top-20 KPI counts.\n"
    "- trace.csv: 34,228 rows; time range 2022-03-20 09:30 → 2022-03-20 09:59; 40 unique cmdb_id samples. "
    "Top traces include trace.from_frontend-2.* (row_count, error_rate, duration_p95, duration_mean at 608) and similar frontend traces. "
    "See trace_name_top20 for details.\n"
    "- log.csv: 1,824 rows; time range 2022-03-20 09:30 → 2022-03-20 10:00; 31 unique cmdb_id samples. "
    "log.error_count and log.row_count both have 912 entries each (see log_name_top20).\n"
    "- error_logs.csv: 0 rows (empty); no timestamps, no cmdb_id, no messages (error_top20 is empty).\n\n"
    "Full datasets remain available in metric_df, trace_df, log_df, error_df. "
    "Compact top-count summaries are in metric_kpi_top20, trace_name_top20, log_name_top20, and error_top20."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(rows                                                                   319703
min_timestamp                                       2022-03-20 09:00:00+08:00
max_timestamp                                       2022-03-20 10:00:00+08:00
unique_cmdb_sample_count                                                   50
unique_cmdb_sample          [adservice, adservice-0, adservice-1, adservic...
dtype: object,                                              kpi_name  count
0   container.node-6.container_network_receive_pac...   2196
1   container.node-6.container_network_transmit_pa...   2196
2   container.node-6.container_network_receive_MB....   2196
3   container.node-6.container_network_receive_pac...   2196
4   container.node-6.container_network_receive_err...   2196
5   container.node-6.container_network_transmit_er...   2196
6   container.node-6.container_network_transmit_pa...   2196
7   container.node-6.container_network_transmit_MB...   2196
8              container.node-6.container_memory_swap   2074
9       container.node-6.container_fs_reads./dev/vda1   2074
10   container.node-6.container_memory_working_set_MB   2074
11     container.node-6.container_fs_inodes./dev/vda1   2074
12        container.node-6.container_file_descriptors   2074
13     container.node-6.container_memory_max_usage_MB   2074
14              container.node-6.container_memory_rss   2074
15         container.node-6.container_memory_usage_MB   2074
16    container.node-6.container_tasks_state.sleeping   2074
17     container.node-6.container_tasks_state.running   2074
18   container.node-6.container_tasks_state.iowaiting   2074
19  container.node-6.container_fs_sector_reads./de...   2074, rows                                                                    34228
min_timestamp                                       2022-03-20 09:30:00+08:00
max_timestamp                                       2022-03-20 09:59:00+08:00
unique_cmdb_sample_count                                                   40
unique_cmdb_sample          [adservice-0, adservice-1, adservice-2, adserv...
dtype: object,                                     trace_name  count
0              trace.from_frontend-2.row_count    608
1             trace.from_frontend-2.error_rate    608
2           trace.from_frontend-2.duration_p95    608
3          trace.from_frontend-2.duration_mean    608
4              trace.from_frontend-1.row_count    603
5             trace.from_frontend-1.error_rate    603
6           trace.from_frontend-1.duration_p95    603
7          trace.from_frontend-1.duration_mean    603
8          trace.from_frontend-0.duration_mean    600
9           trace.from_frontend-0.duration_p95    600
10            trace.from_frontend-0.error_rate    600
11             trace.from_frontend-0.row_count    600
12                        trace.self.row_count    477
13                       trace.self.error_rate    477
14                     trace.self.duration_p95    477
15                    trace.self.duration_mean    477
16      trace.from_checkoutservice-1.row_count    467
17     trace.from_checkoutservice-1.error_rate    467
18   trace.from_checkoutservice-1.duration_p95    467
19  trace.from_checkoutservice-1.duration_mean    467, rows                                                                     1824
min_timestamp                                       2022-03-20 09:30:00+08:00
max_timestamp                                       2022-03-20 10:00:00+08:00
unique_cmdb_sample_count                                                   31
unique_cmdb_sample          [adservice-0, adservice-1, adservice-2, cartse...
dtype: object,           log_name  count
0  log.error_count    912
1    log.row_count    912, rows                          0
min_timestamp               NaT
max_timestamp               NaT
unique_cmdb_sample_count      0
unique_cmdb_sample           []
dtype: object, Empty DataFrame
Columns: [dt, cmdb_id, message_snip]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 and P5 per (cmdb_id, kpi_name) using full metric_df
grp = metric_df.groupby(['cmdb_id', 'kpi_name'])['value']
p95 = grp.quantile(0.95).rename('p95').reset_index()
p5 = grp.quantile(0.05).rename('p5').reset_index()
metric_thresholds_df = p95.merge(p5, on=['cmdb_id', 'kpi_name'])

# 2) Filter metric_df to the incident window (inclusive)
start = pd.Timestamp('2022-03-20 09:30:00').tz_localize(tz)
end = pd.Timestamp('2022-03-20 10:00:00').tz_localize(tz)
metric_window_df = metric_df[(metric_df['dt'] >= start) & (metric_df['dt'] <= end)].copy()

# 3) Join windowed data with thresholds and mark anomalies
metric_window_anom_df = metric_window_df.merge(metric_thresholds_df, on=['cmdb_id', 'kpi_name'], how='left')
metric_window_anom_df['anomaly_high'] = metric_window_anom_df['value'] >= metric_window_anom_df['p95']
metric_window_anom_df['anomaly_low'] = metric_window_anom_df['value'] <= metric_window_anom_df['p5']
metric_window_anom_df['any_anom'] = metric_window_anom_df['anomaly_high'] | metric_window_anom_df['anomaly_low']

# 4) Aggregate to produce anomaly_candidates_df
group_cols = ['cmdb_id', 'kpi_name']

total_rows = metric_window_anom_df.groupby(group_cols)['value'].count().rename('total_rows_in_window')
max_value = metric_window_anom_df.groupby(group_cols)['value'].max().rename('max_value_in_window')
min_value = metric_window_anom_df.groupby(group_cols)['value'].min().rename('min_value_in_window')
p95_series = metric_window_anom_df.groupby(group_cols)['p95'].first().rename('p95')
p5_series = metric_window_anom_df.groupby(group_cols)['p5'].first().rename('p5')
anomaly_count = metric_window_anom_df.groupby(group_cols)['any_anom'].sum().astype(int).rename('anomaly_count')

# earliest anomaly timestamp per group (only from rows flagged as anomalies)
anom_rows = metric_window_anom_df[metric_window_anom_df['any_anom']]
if not anom_rows.empty:
    earliest_anom = anom_rows.groupby(group_cols)['dt'].min().rename('earliest_anomaly_timestamp')
else:
    # empty series with same MultiIndex structure if no anomalies
    earliest_anom = pd.Series(dtype='datetime64[ns, Asia/Shanghai]')

# combine all aggregates
anomaly_candidates_df = pd.concat(
    [anomaly_count, earliest_anom, total_rows, p95_series, p5_series, max_value, min_value],
    axis=1
).reset_index()

# Ensure earliest_anomaly_timestamp column exists
if 'earliest_anomaly_timestamp' not in anomaly_candidates_df.columns:
    anomaly_candidates_df['earliest_anomaly_timestamp'] = pd.NaT

# 5) Prepare compact outputs: top 20 by anomaly_count desc, and counts
anomaly_candidates_top20 = anomaly_candidates_df.sort_values('anomaly_count', ascending=False).head(20)[
    ['cmdb_id', 'kpi_name', 'anomaly_count', 'earliest_anomaly_timestamp',
     'total_rows_in_window', 'p95', 'p5', 'max_value_in_window', 'min_value_in_window']
]

total_pairs_scanned = anomaly_candidates_df.shape[0]  # number of (cmdb_id,kpi_name) pairs present in the window
pairs_with_anom = (anomaly_candidates_df['anomaly_count'] > 0).sum()

# Return compact results
anomaly_candidates_top20, total_pairs_scanned, pairs_with_anom

```
Out[2]:
```
summary = (
    "Actions performed:\n"
    "- Computed global P95 and P5 per (cmdb_id, kpi_name) into metric_thresholds_df.\n"
    "- Filtered metric_df to 2022-03-20 09:30 → 2022-03-20 10:00 (metric_window_df).\n"
    "- Joined windowed data with thresholds and flagged anomalies where value >= P95 or value <= P5 (metric_window_anom_df).\n"
    "- Aggregated results into anomaly_candidates_df and selected top candidates.\n\n"
    "Key findings (compact):\n"
    "- Top 20 candidates all have anomaly_count = 31 (an anomaly at every sample in the 31-minute window), earliest anomaly = 2022-03-20 09:30+08, total_rows_in_window = 31.\n"
    "- Frequently appearing components in the top list: shippingservice2-0 (many container.node-6 metrics), node-5 (system.swap, system.tcp.retrans_pct), and adservice (runtime.ClassLoading_LoadedClassCount).\n"
    "- Totals: scanned (cmdb_id, kpi_name) pairs = 5253; pairs with anomaly_count > 0 = 5244 (nearly all pairs flagged).\n\n"
    "Note: The very high number of flagged pairs suggests the thresholds or anomaly directionality should be reviewed (many metrics are constant or equal to their P5/P95), before concluding a widespread real incident."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                 cmdb_id                                           kpi_name  anomaly_count earliest_anomaly_timestamp  total_rows_in_window           p95            p5  max_value_in_window  min_value_in_window
8              adservice    runtime.java_lang_ClassLoading_LoadedClassCount             31  2022-03-20 09:30:00+08:00                    31  5.162000e+03  5.162000e+03         5.162000e+03         5.162000e+03
5250  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....             31  2022-03-20 09:30:00+08:00                    31  1.050000e-03  0.000000e+00         1.050000e-03         0.000000e+00
5209  shippingservice2-0              container.node-6.container_memory_rss             31  2022-03-20 09:30:00+08:00                    31  1.110835e+07  1.110835e+07         1.110835e+07         1.110835e+07
5225  shippingservice2-0    container.node-6.container_spec_memory_limit_MB             31  2022-03-20 09:30:00+08:00                    31  1.280000e+02  1.280000e+02         1.280000e+02         1.280000e+02
5226  shippingservice2-0  container.node-6.container_spec_memory_reserva...             31  2022-03-20 09:30:00+08:00                    31  0.000000e+00  0.000000e+00         0.000000e+00         0.000000e+00
5227  shippingservice2-0  container.node-6.container_spec_memory_swap_li...             31  2022-03-20 09:30:00+08:00                    31  1.280000e+02  1.280000e+02         1.280000e+02         1.280000e+02
5228  shippingservice2-0      container.node-6.container_start_time_seconds             31  2022-03-20 09:30:00+08:00                    31  1.647144e+09  1.647144e+09         1.647144e+09         1.647144e+09
5229  shippingservice2-0   container.node-6.container_tasks_state.iowaiting             31  2022-03-20 09:30:00+08:00                    31  0.000000e+00  0.000000e+00         0.000000e+00         0.000000e+00
5230  shippingservice2-0     container.node-6.container_tasks_state.running             31  2022-03-20 09:30:00+08:00                    31  0.000000e+00  0.000000e+00         0.000000e+00         0.000000e+00
5231  shippingservice2-0    container.node-6.container_tasks_state.sleeping             31  2022-03-20 09:30:00+08:00                    31  0.000000e+00  0.000000e+00         0.000000e+00         0.000000e+00
5232  shippingservice2-0     container.node-6.container_tasks_state.stopped             31  2022-03-20 09:30:00+08:00                    31  0.000000e+00  0.000000e+00         0.000000e+00         0.000000e+00
3687              node-5                                     system.swap.si             31  2022-03-20 09:30:00+08:00                    31  0.000000e+00  0.000000e+00         0.000000e+00         0.000000e+00
3688              node-5                                     system.swap.so             31  2022-03-20 09:30:00+08:00                    31  0.000000e+00  0.000000e+00         0.000000e+00         0.000000e+00
5233  shippingservice2-0  container.node-6.container_tasks_state.uninter...             31  2022-03-20 09:30:00+08:00                    31  0.000000e+00  0.000000e+00         0.000000e+00         0.000000e+00
5234  shippingservice2-0                 container.node-6.container_threads             31  2022-03-20 09:30:00+08:00                    31  2.300000e+01  2.300000e+01         2.300000e+01         2.300000e+01
5235  shippingservice2-0             container.node-6.container_threads_max             31  2022-03-20 09:30:00+08:00                    31  0.000000e+00  0.000000e+00         0.000000e+00         0.000000e+00
5236  shippingservice2-0  container.node-6.container_ulimits_soft.max_op...             31  2022-03-20 09:30:00+08:00                    31  1.048576e+06  1.048576e+06         1.048576e+06         1.048576e+06
5238  shippingservice2-0  mesh.destination.checkoutservice2.shippingserv...             31  2022-03-20 09:30:00+08:00                    31  5.250000e-04  0.000000e+00         1.050000e-03         0.000000e+00
3692              node-5                             system.tcp.retrans_pct             31  2022-03-20 09:30:00+08:00                    31  1.000000e-02  1.000000e-02         1.000000e-02         1.000000e-02
3693              node-5                             system.udp.connect.num             31  2022-03-20 09:30:00+08:00                    31  1.100000e+01  1.100000e+01         1.200000e+01         1.100000e+01, 5253, np.int64(5244))```
```

In [ ]:
# In[3]:

import numpy as np
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse anomaly_candidates_df from kernel

# 1) Filter out constant metrics where max == min
non_constant_df = anomaly_candidates_df[anomaly_candidates_df['max_value_in_window'] != anomaly_candidates_df['min_value_in_window']].copy()

# 2) Exclude KPIs containing any of the specified substrings
exclude_subs = [
    'mesh.', 'container_spec', 'container_start_time_seconds', 'ulimits', 'tasks_state',
    'threads_max', 'file_descriptors', 'inodes', 'fs_sector', 'container_threads_max',
    'container_start_time', 'container_fs_inodes', 'container_fs_sector_reads'
]

def contains_excluded(s):
    return any(sub in s for sub in exclude_subs)

mask_exclude = non_constant_df['kpi_name'].apply(lambda s: contains_excluded(s))
true_candidate_df = non_constant_df[~mask_exclude].copy()

# 3) Compute per-cmdb_id aggregates

# Ensure earliest_anomaly_timestamp is datetime tz-aware; leave as-is (already tz-aware)
# Compute severity per KPI
eps = 1e-9
p95 = true_candidate_df['p95'].astype(float)
p5 = true_candidate_df['p5'].astype(float)
maxv = true_candidate_df['max_value_in_window'].astype(float)
minv = true_candidate_df['min_value_in_window'].astype(float)

sev_high = (maxv - p95) / np.maximum(np.abs(p95), eps)
sev_low = (p5 - minv) / np.maximum(np.abs(p5), eps)
true_candidate_df['severity'] = np.maximum(sev_high.fillna(0), sev_low.fillna(0))

# For faulty_kpis_count we consider KPIs with anomaly_count>0
# Prepare grouping
group = true_candidate_df.groupby('cmdb_id')

# faulty_kpis_count: count of distinct kpi_name where anomaly_count>0
faulty_counts = group.apply(lambda g: g.loc[g['anomaly_count']>0, 'kpi_name'].nunique()).rename('faulty_kpis_count')

# earliest_anomaly: min earliest_anomaly_timestamp across its KPIs (skip NaT)
earliest_anom = group['earliest_anomaly_timestamp'].min().rename('earliest_anomaly')

# top_kpis: top 5 kpi_name by anomaly_count for that cmdb_id (as list)
def top_kpis_list(g, topn=5):
    s = g.sort_values('anomaly_count', ascending=False).head(topn)
    return s['kpi_name'].tolist()

top_kpis = group.apply(lambda g: top_kpis_list(g, 5)).rename('top_kpis')

# max_severity: max of severity across KPIs
max_sev = group['severity'].max().rename('max_severity')

# Combine into per_component_df
per_component_df = pd.concat([faulty_counts, earliest_anom, max_sev, top_kpis], axis=1).reset_index()

# Replace NaT in earliest_anomaly with NaT (keeps dtype); ensure sorted
per_component_df['earliest_anomaly'] = per_component_df['earliest_anomaly'].dt.tz_convert(tz)

# 4) Prepare final outputs: top 20 cmdb_id rows sorted by faulty_kpis_count desc
per_component_top20 = per_component_df.sort_values('faulty_kpis_count', ascending=False).head(20)[
    ['cmdb_id', 'faulty_kpis_count', 'earliest_anomaly', 'max_severity', 'top_kpis']
]

# Counts to return
total_pairs_scanned_original = anomaly_candidates_df.shape[0]
pairs_remaining_after_filter = true_candidate_df.shape[0]
cmdb_id_count_in_result = per_component_df.shape[0]

# Return compact outputs (variables)
true_candidate_df, per_component_top20, total_pairs_scanned_original, pairs_remaining_after_filter, cmdb_id_count_in_result

```
Out[3]:
```
Summary (plain English):

- Starting point: 5,253 (cmdb_id, kpi_name) pairs were scanned from anomaly_candidates_df.
- After removing constant metrics and excluding meta/static/mesh KPIs, 936 pairs remain (true_candidate_df).
- These 936 pairs cover 59 distinct cmdb_id components (per_component_df).

Top findings (by number of faulty KPIs per component):
- adservice: 53 faulty KPIs, earliest anomaly 2022-03-20 09:30+08, max_severity ≈ 1.0
- adservice2: 49 faulty KPIs, earliest anomaly 2022-03-20 09:30+08, max_severity ≈ 7.5e8
- node-5: 37 faulty KPIs, earliest anomaly 2022-03-20 09:30+08, max_severity ≈ 4.0e9
- node-6: 36 faulty KPIs, earliest anomaly 2022-03-20 09:30+08, max_severity ≈ 0.95
- node-1: 35 faulty KPIs, earliest anomaly 2022-03-20 09:30+08, max_severity ≈ 103.28
- node-2: 34 faulty KPIs, earliest anomaly 2022-03-20 09:30+08, max_severity ≈ 0.758
- node-4: 33 faulty KPIs, earliest anomaly 2022-03-20 09:30+08, max_severity ≈ 1.4
- node-3: 33 faulty KPIs, earliest anomaly 2022-03-20 09:30+08, max_severity ≈ 3.0
- cartservice2-0: 27 faulty KPIs, earliest anomaly 09:30+08, very large max_severity reported
- emailservice-0: 26 faulty KPIs, earliest anomaly 09:30+08, very large max_severity reported

- Common top KPI themes (from top_kpis lists): many container- and system-level metrics (e.g., container CPU throttling, disk/io metrics, process/file descriptors, system CPU/io metrics) are among the top KPIs per component.

Notes / interpretation:
- The earliest anomalies for the top components all start at 2022-03-20 09:30+08.
- Several components show extremely large "max_severity" values (orders of magnitude), indicating very large relative deviations for at least one KPI; this may merit focused inspection, but can also be caused by small denominators (very small p95/p5) amplifying the ratio.
- Next actions: inspect the top_kpis for the top components (especially adservice/adservice2 and node-5) and examine raw time series and logs for those KPIs to confirm real faults vs. metric artifacts.

The original code execution output of IPython Kernel is also provided below for reference:

(                 cmdb_id                                           kpi_name  anomaly_count earliest_anomaly_timestamp  total_rows_in_window           p95            p5  max_value_in_window  min_value_in_window  severity
0              adservice                                     app.grpc.count              5  2022-03-20 09:53:00+08:00                    31    157.000000     74.000000           171.000000            96.000000  0.089172
1              adservice                                       app.grpc.mrt              3  2022-03-20 09:36:00+08:00                    31      0.002590      0.002286             0.002644             0.002274  0.020555
5              adservice                                       app.http.mrt              6  2022-03-20 09:31:00+08:00                    31      0.008931      0.008500             0.008973             0.008440  0.007108
13             adservice  runtime.java_lang_Compilation_TotalCompilation...             30  2022-03-20 09:30:00+08:00                    31  47014.000000  47013.000000         47014.000000         47013.000000  0.000000
14             adservice  runtime.java_lang_GarbageCollector_CollectionC...              4  2022-03-20 09:57:00+08:00                    31  34911.416667  34766.166667         34919.583333         34838.666667  0.000234
...                  ...                                                ...            ...                        ...                   ...           ...           ...                  ...                  ...       ...
5212  shippingservice2-0   container.node-6.container_memory_working_set_MB              9  2022-03-20 09:52:00+08:00                    31     23.515625     23.400391            23.568359            23.500000  0.002243
5213  shippingservice2-0  container.node-6.container_network_receive_MB....              4  2022-03-20 09:38:00+08:00                    31      0.034823      0.021961             0.329698             0.021961  8.467712
5215  shippingservice2-0  container.node-6.container_network_receive_pac...              3  2022-03-20 09:38:00+08:00                    31    363.000000    266.000000           480.000000           257.500000  0.322314
5217  shippingservice2-0  container.node-6.container_network_transmit_MB...              3  2022-03-20 09:39:00+08:00                    31      0.461197      0.424775             0.487634             0.431636  0.057321
5219  shippingservice2-0  container.node-6.container_network_transmit_pa...              4  2022-03-20 09:38:00+08:00                    31    233.500000    158.000000           339.500000           157.000000  0.453961

[936 rows x 10 columns],                     cmdb_id  faulty_kpis_count          earliest_anomaly  max_severity                                           top_kpis
0                 adservice                 53 2022-03-20 09:30:00+08:00  1.000000e+00  [runtime.java_lang_GarbageCollector_LastGcInfo...
4                adservice2                 49 2022-03-20 09:30:00+08:00  7.500000e+08  [runtime.java_lang_GarbageCollector_LastGcInfo...
35                   node-5                 37 2022-03-20 09:30:00+08:00  4.000000e+09  [system.cpu.iowait, system.process.zombie.num,...
36                   node-6                 36 2022-03-20 09:30:00+08:00  9.541284e-01  [system.io.r_await, system.disk.total, system....
31                   node-1                 35 2022-03-20 09:30:00+08:00  1.032778e+02  [system.process.zombie.num, system.io.avg_q_sz...
32                   node-2                 34 2022-03-20 09:30:00+08:00  7.582160e-01  [system.process.zombie.num, system.os.nofile.u...
34                   node-4                 33 2022-03-20 09:30:00+08:00  1.400000e+00  [system.io.await, system.io.w_s, system.io.svc...
33                   node-3                 33 2022-03-20 09:30:00+08:00  3.000000e+00  [system.os.nofile.used_pct, system.io.avg_q_sz...
10           cartservice2-0                 27 2022-03-20 09:30:00+08:00  1.100000e+11  [container.node-5.container_cpu_cfs_throttled_...
22           emailservice-0                 26 2022-03-20 09:30:00+08:00  1.791000e+12  [container.node-6.container_cpu_cfs_throttled_...
7             cartservice-0                 19 2022-03-20 09:30:00+08:00  5.000000e+08  [container.node-6.container_cpu_cfs_throttled_...
14        checkoutservice-2                 19 2022-03-20 09:30:00+08:00  3.300000e+10  [container.node-5.container_memory_cache, cont...
9             cartservice-2                 19 2022-03-20 09:30:00+08:00  1.000000e+09  [container.node-6.container_cpu_cfs_throttled_...
18        currencyservice-1                 17 2022-03-20 09:30:00+08:00  2.169498e+00  [container.node-6.container_cpu_cfs_throttled_...
17        currencyservice-0                 17 2022-03-20 09:30:00+08:00  7.410768e+00  [container.node-6.container_cpu_cfs_throttled_...
20       currencyservice2-0                 17 2022-03-20 09:30:00+08:00  6.619469e+00  [container.node-6.container_cpu_cfs_throttled_...
19        currencyservice-2                 17 2022-03-20 09:30:00+08:00  1.890219e+00  [container.node-6.container_cpu_cfs_throttled_...
1               adservice-0                 16 2022-03-20 09:30:00+08:00  9.254239e+00  [container.node-6.container_threads, container...
2               adservice-1                 16 2022-03-20 09:30:00+08:00  4.954466e+00  [container.node-6.container_threads, container...
44  productcatalogservice-1                 16 2022-03-20 09:30:00+08:00  1.650000e+10  [container.node-6.container_cpu_cfs_throttled_..., 5253, 936, 59)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 and P5 per (cmdb_id, trace_name) using full trace_df
trace_grp = trace_df.groupby(['cmdb_id', 'trace_name'])['value']
trace_p95 = trace_grp.quantile(0.95).rename('p95').reset_index()
trace_p5 = trace_grp.quantile(0.05).rename('p5').reset_index()
trace_thresholds_df = trace_p95.merge(trace_p5, on=['cmdb_id', 'trace_name'])

# 2) Filter trace_df to incident window (inclusive)
start = pd.Timestamp('2022-03-20 09:30:00').tz_localize(tz)
end = pd.Timestamp('2022-03-20 10:00:00').tz_localize(tz)
trace_window_df = trace_df[(trace_df['dt'] >= start) & (trace_df['dt'] <= end)].copy()

# 3) Define anomaly rules and 4) join thresholds and flag anomalies
trace_window_anom_df = trace_window_df.merge(trace_thresholds_df, on=['cmdb_id', 'trace_name'], how='left')

def flag_anom(row):
    name = row['trace_name']
    val = row['value']
    p95 = row['p95']
    p5 = row['p5']
    # default high-anomaly
    if 'error_rate' in name:
        return bool(val >= p95)
    if 'row_count' in name:
        return bool(val <= p5)
    if 'duration' in name:
        return bool(val >= p95)
    # fallback high-anomaly
    return bool(val >= p95)

trace_window_anom_df['is_anom'] = trace_window_anom_df.apply(flag_anom, axis=1)

# 5) Aggregate by cmdb_id
# anomaly_count: number of anomalous rows
anom_rows = trace_window_anom_df[trace_window_anom_df['is_anom']].copy()

# distinct_trace_names_count per cmdb_id (count of distinct trace_name that had any anomaly)
distinct_trace_names_count = anom_rows.groupby('cmdb_id')['trace_name'].nunique().rename('distinct_trace_names_count')

# anomaly_count per cmdb_id (number of anomalous rows)
anomaly_count = anom_rows.groupby('cmdb_id').size().rename('anomaly_count')

# earliest_anomaly_timestamp per cmdb_id
if not anom_rows.empty:
    earliest_anomaly_timestamp = anom_rows.groupby('cmdb_id')['dt'].min().rename('earliest_anomaly_timestamp')
else:
    earliest_anomaly_timestamp = pd.Series(dtype='datetime64[ns, Asia/Shanghai]')

# list_top_trace_names: top 5 trace_name by anomaly occurrences per cmdb_id
def top_trace_names(series, topn=5):
    return series.value_counts().head(topn).index.tolist()

top_names = anom_rows.groupby('cmdb_id')['trace_name'].apply(lambda s: top_trace_names(s, 5)).rename('list_top_trace_names')

# Combine aggregates
trace_anomaly_agg_df = pd.concat([anomaly_count, distinct_trace_names_count, earliest_anomaly_timestamp, top_names], axis=1).reset_index()
# Ensure columns exist even if no anomalies
for col in ['anomaly_count', 'distinct_trace_names_count', 'earliest_anomaly_timestamp', 'list_top_trace_names']:
    if col not in trace_anomaly_agg_df.columns:
        trace_anomaly_agg_df[col] = pd.NA

# 6) Prepare outputs
trace_anomaly_top20 = trace_anomaly_agg_df.sort_values('anomaly_count', ascending=False).head(20)[
    ['cmdb_id', 'anomaly_count', 'distinct_trace_names_count', 'earliest_anomaly_timestamp', 'list_top_trace_names']
]

total_pairs_scanned = trace_thresholds_df.shape[0]
pairs_with_anom = anom_rows.groupby(['cmdb_id', 'trace_name']).ngroups if not anom_rows.empty else 0

high_metric_candidates = ['adservice','adservice2','node-5','node-6','node-1','node-2','node-3','node-4','cartservice2-0','emailservice-0','shippingservice2-0']
# intersection between cmdb_id values that had anomalies and the high-metric candidates
anomalous_cmdbs = trace_anomaly_agg_df['cmdb_id'].tolist()
intersection_list = [c for c in high_metric_candidates if c in anomalous_cmdbs]

# Return compact aggregates
trace_anomaly_top20, total_pairs_scanned, pairs_with_anom, intersection_list

```
Out[4]:
```
summary = (
    "Trace anomaly summary (timestamps in UTC+8):\n"
    "- P95/P5 computed for 1,236 (cmdb_id, trace_name) pairs.\n"
    "- In the incident window (2022-03-20 09:30 → 10:00), all 1,236 pairs produced anomalies (pairs_with_anom = 1,236).\n"
    "- Top affected components (by anomaly_count): checkoutservice-0 (1,135), checkoutservice-1 (1,122), checkoutservice-2 (1,053), frontend-1 (914), frontend-2 (902), frontend-0 (886). Earliest anomalies for top components start at 2022-03-20 09:30+08.\n"
    "- Many components show large numbers of distinct trace anomalies (e.g., 92 distinct trace_names for checkout and frontend pods). The most common anomalous trace types are error_rate and duration traces (e.g., trace.from_root.error_rate, trace.self.error_rate, trace.from_frontend-*.error_rate).\n"
    "- Intersection with high-metric candidates: ['cartservice2-0', 'emailservice-0', 'shippingservice2-0'] — these three appear in both the trace-anomaly results and the earlier high-metric candidate list.\n"
    "Interpretation: There is broad trace-level instability starting at 09:30+08, dominated by elevated error rates and latency traces across many frontend/checkout components. Focused investigation should target checkout/frontend clusters and the three overlapping components from the metric analysis (cartservice2-0, emailservice-0, shippingservice2-0)."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                    cmdb_id  anomaly_count  distinct_trace_names_count earliest_anomaly_timestamp                               list_top_trace_names
8         checkoutservice-0           1135                          92  2022-03-20 09:30:00+08:00  [trace.from_root.error_rate, trace.self.error_...
9         checkoutservice-1           1122                          92  2022-03-20 09:30:00+08:00  [trace.from_root.error_rate, trace.self.error_...
10        checkoutservice-2           1053                          92  2022-03-20 09:30:00+08:00  [trace.from_root.error_rate, trace.self.error_...
21               frontend-1            914                          92  2022-03-20 09:30:00+08:00  [trace.from_root.error_rate, trace.self.error_...
22               frontend-2            902                          92  2022-03-20 09:30:00+08:00  [trace.from_root.error_rate, trace.self.error_...
20               frontend-0            886                          92  2022-03-20 09:30:00+08:00  [trace.from_root.error_rate, trace.self.error_...
4             cartservice-0            488                          28  2022-03-20 09:30:00+08:00  [trace.from_frontend-1.error_rate, trace.from_...
6             cartservice-2            479                          28  2022-03-20 09:30:00+08:00  [trace.from_frontend-0.error_rate, trace.from_...
28  productcatalogservice-0            395                          40  2022-03-20 09:30:00+08:00  [trace.from_frontend-0.error_rate, trace.from_...
30  productcatalogservice-2            393                          40  2022-03-20 09:30:00+08:00  [trace.from_frontend-0.error_rate, trace.from_...
29  productcatalogservice-1            389                          44  2022-03-20 09:30:00+08:00  [trace.from_frontend-0.error_rate, trace.from_...
11       checkoutservice2-0            378                          36  2022-03-20 09:30:00+08:00  [trace.from_frontend2-0.error_rate, trace.from...
23              frontend2-0            368                          36  2022-03-20 09:30:00+08:00  [trace.from_root.error_rate, trace.self.error_...
5             cartservice-1            368                          28  2022-03-20 09:30:00+08:00  [trace.from_frontend-0.error_rate, trace.self....
32  recommendationservice-0            271                          28  2022-03-20 09:30:00+08:00  [trace.from_frontend-0.error_rate, trace.from_...
33  recommendationservice-1            264                          28  2022-03-20 09:30:00+08:00  [trace.from_frontend-0.error_rate, trace.from_...
34  recommendationservice-2            257                          28  2022-03-20 09:30:00+08:00  [trace.from_frontend-0.error_rate, trace.from_...
37        shippingservice-1            253                          24  2022-03-20 09:30:00+08:00  [trace.from_checkoutservice-0.error_rate, trac...
36        shippingservice-0            243                          24  2022-03-20 09:30:00+08:00  [trace.from_frontend-1.error_rate, trace.from_...
38        shippingservice-2            239                          24  2022-03-20 09:30:00+08:00  [trace.from_frontend-2.error_rate, trace.from_..., 1236, 1236, ['cartservice2-0', 'emailservice-0', 'shippingservice2-0'])```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Candidates
candidates = ['cartservice2-0','emailservice-0','shippingservice2-0']

# Time window (UTC+8)
start = pd.Timestamp('2022-03-20 09:30:00').tz_localize(tz)
end = pd.Timestamp('2022-03-20 10:00:00').tz_localize(tz)

# 1) Log summary for candidates in the time window
# Ensure log_df has dt column (it should from earlier steps)
log_window = log_df[(log_df['dt'] >= start) & (log_df['dt'] <= end) & (log_df['cmdb_id'].isin(candidates))].copy()

# Coerce value to numeric for summation (non-numeric -> NaN)
if 'value' in log_window.columns:
    log_window['value_num'] = pd.to_numeric(log_window['value'], errors='coerce')
else:
    log_window['value_num'] = np.nan

log_summary_df = (
    log_window
    .groupby(['cmdb_id', 'log_name'], dropna=False)
    .agg(
        row_count = ('log_name', 'count'),
        sum_of_value = ('value_num', 'sum'),
        earliest_log_timestamp = ('dt', 'min'),
        latest_log_timestamp = ('dt', 'max')
    )
    .reset_index()
)

# 2) Top metric anomalies per candidate using true_candidate_df (already filtered non-constant & excluded)
# Ensure true_candidate_df exists in kernel
tc = true_candidate_df.copy()
tc_candidates = tc[tc['cmdb_id'].isin(candidates)].copy()

# Select desired columns and sort by severity desc, then take top 10 per cmdb_id
cols_needed = ['cmdb_id', 'kpi_name', 'anomaly_count', 'earliest_anomaly_timestamp',
               'total_rows_in_window', 'p95', 'p5', 'max_value_in_window', 'min_value_in_window', 'severity']

# Some rows might lack severity column if upstream changed; ensure exists
if 'severity' not in tc_candidates.columns:
    # compute a fallback severity if possible
    eps = 1e-9
    p95s = tc_candidates['p95'].astype(float)
    p5s = tc_candidates['p5'].astype(float)
    maxv = tc_candidates['max_value_in_window'].astype(float)
    minv = tc_candidates['min_value_in_window'].astype(float)
    sev_high = (maxv - p95s) / np.maximum(np.abs(p95s), eps)
    sev_low = (p5s - minv) / np.maximum(np.abs(p5s), eps)
    tc_candidates['severity'] = np.maximum(sev_high.fillna(0), sev_low.fillna(0))

# Ensure all columns exist
for c in cols_needed:
    if c not in tc_candidates.columns:
        tc_candidates[c] = pd.NA

# Get top 10 per candidate
metric_top_list = []
for cm in candidates:
    sub = tc_candidates[tc_candidates['cmdb_id'] == cm].sort_values('severity', ascending=False).head(10)
    metric_top_list.append(sub[cols_needed])
metric_top_anoms_df = pd.concat(metric_top_list, ignore_index=True)

# Limit total returned rows to 30 just in case
metric_top_anoms_df = metric_top_anoms_df.head(30)

# 3) Earliest anomaly timestamp among the returned metric anomalies (single value)
if not metric_top_anoms_df['earliest_anomaly_timestamp'].dropna().empty:
    earliest_metric_anom_timestamp = metric_top_anoms_df['earliest_anomaly_timestamp'].min()
else:
    earliest_metric_anom_timestamp = pd.NaT

# Store subsets in kernel variables for follow-up steps
selected_log_window = log_window  # full filtered log rows for these candidates in window
selected_metric_top = metric_top_anoms_df  # selected top metric anomalies for candidates

# Return compact aggregates as requested (variables)
log_summary_df, metric_top_anoms_df, earliest_metric_anom_timestamp

```
Out[5]:
```
Summary for the three overlapping candidates (cartservice2-0, emailservice-0, shippingservice2-0) — compact, plain English:

1) Logs (2022-03-20 09:30 → 10:00, UTC+8)
- emailservice-0:
  - log.error_count: row_count = 30, sum_of_value = 0.0, earliest = 2022-03-20 09:30:00+08:00, latest = 2022-03-20 09:59:00+08:00
  - log.row_count:   row_count = 30, sum_of_value = 194.0, earliest = 2022-03-20 09:30:00+08:00, latest = 2022-03-20 09:59:00+08:00
- cartservice2-0 and shippingservice2-0: no log rows in the window (none returned).

2) Top metric anomalies (up to 10 per component; returned 30 rows total)
- cartservice2-0 (top examples, very high severities):
  - container.node-5.container_memory_failures.* (anomaly_count=31, earliest=09:30, severity ≈ 1.1e11)
  - container.node-5.container_fs_writes./dev/vda (anomaly_count=31, severity ≈ 3.0e10)
  - container.node-5.container_cpu_cfs_throttled_* (anomaly_count=31, severities ≈ 2.4e10 and 3.13e9)
  - (These indicate extremely large relative deviations for container memory failures, disk writes and CPU throttling.)
- shippingservice2-0 (examples, moderate-to-low severities):
  - container.node-6.container_network_receive_MB... (severity ≈ 8.47)
  - container.node-6.container_network_receive_packets... (severity ≈ 0.322)
  - container.node-6.container_network_transmit_MB... (severity ≈ 0.057)
  - container.node-6.container_memory_working_set_MB, container_memory_usage_MB, container_fs_usage_MB (small severities ~1e-3 to 1e-4)
  - (These point to network and memory-related anomalies on node-6 containers for shippingservice2-0.)
- emailservice-0:
  - (Log entries exist; metric anomalies for emailservice-0 were part of the candidate set previously and showed high severities in earlier aggregates. In this top-30 slice the prominent metric rows shown are mostly for cartservice2-0 and shippingservice2-0; emailservice-0 had logs and likely metric anomalies in the full set.)

3) Earliest metric anomaly among these returned metric anomalies:
- 2022-03-20 09:30:00+08:00

Stored kernel variables for follow-up:
- selected_log_window (filtered log rows for the three candidates in window)
- selected_metric_top (selected top metric anomalies returned)

Interpretation / next steps (brief):
- cartservice2-0 shows extreme metric severity on container memory failures, CPU throttling and disk writes — investigate container/node memory and I/O on node-5 hosting this pod.
- shippingservice2-0 shows network and memory anomalies on node-6 containers — inspect node-6 networking and container resource usage.
- emailservice-0 has log evidence (row_count and error_count) in the window; combine its logs with its metric anomalies for deeper triage.
- All metric anomalies for these candidates begin at 09:30+08; focus investigation around that time.

The original code execution output of IPython Kernel is also provided below for reference:

(          cmdb_id         log_name  row_count  sum_of_value    earliest_log_timestamp      latest_log_timestamp
0  emailservice-0  log.error_count         30           0.0 2022-03-20 09:30:00+08:00 2022-03-20 09:59:00+08:00
1  emailservice-0    log.row_count         30         194.0 2022-03-20 09:30:00+08:00 2022-03-20 09:59:00+08:00,                cmdb_id                                           kpi_name  anomaly_count earliest_anomaly_timestamp  total_rows_in_window        p95         p5  max_value_in_window  min_value_in_window      severity
0       cartservice2-0  container.node-5.container_memory_failures.con...             31  2022-03-20 09:30:00+08:00                    31   0.000000   0.000000           110.000000             0.000000  1.100000e+11
1       cartservice2-0  container.node-5.container_memory_failures.hie...             31  2022-03-20 09:30:00+08:00                    31   0.000000   0.000000           110.000000             0.000000  1.100000e+11
2       cartservice2-0      container.node-5.container_fs_writes./dev/vda             31  2022-03-20 09:30:00+08:00                    31   0.000000   0.000000            30.000000             0.000000  3.000000e+10
3       cartservice2-0  container.node-5.container_cpu_cfs_throttled_p...             31  2022-03-20 09:30:00+08:00                    31   0.000000   0.000000            24.000000             0.000000  2.400000e+10
4       cartservice2-0  container.node-5.container_cpu_cfs_throttled_s...             31  2022-03-20 09:30:00+08:00                    31   0.000000   0.000000             3.128959             0.000000  3.128959e+09
..                 ...                                                ...            ...                        ...                   ...        ...        ...                  ...                  ...           ...
25  shippingservice2-0         container.node-6.container_cpu_cfs_periods              4  2022-03-20 09:50:00+08:00                    31  22.500000   2.000000            25.500000             5.000000  1.333333e-01
26  shippingservice2-0  container.node-6.container_network_transmit_MB...              3  2022-03-20 09:39:00+08:00                    31   0.461197   0.424775             0.487634             0.431636  5.732107e-02
27  shippingservice2-0   container.node-6.container_memory_working_set_MB              9  2022-03-20 09:52:00+08:00                    31  23.515625  23.400391            23.568359            23.500000  2.242525e-03
28  shippingservice2-0         container.node-6.container_memory_usage_MB              9  2022-03-20 09:52:00+08:00                    31  32.574219  30.912109            32.626953            32.558594  1.618899e-03
29  shippingservice2-0   container.node-6.container_fs_usage_MB./dev/vda1              4  2022-03-20 09:57:00+08:00                    31  46.025391  45.886719            46.041016            45.927734  3.394865e-04

[30 rows x 10 columns], Timestamp('2022-03-20 09:30:00+0800', tz='Asia/Shanghai'))

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz
from collections import Counter

tz = pytz.timezone('Asia/Shanghai')

# Reuse trace_window_anom_df from kernel
# Ensure 'is_anom' exists and is boolean
if 'is_anom' not in trace_window_anom_df.columns:
    # fallback: consider any non-null 'p95' comparison if present (should not happen normally)
    trace_window_anom_df['is_anom'] = trace_window_anom_df.get('value', pd.Series()).notna()

trace_window_anom_df['is_anom'] = trace_window_anom_df['is_anom'].astype(bool)

candidates = ['cartservice2-0','emailservice-0','shippingservice2-0']

# Pre-filter anomalous rows for efficiency
anom_rows = trace_window_anom_df[trace_window_anom_df['is_anom']].copy()

# Helper to find rows where trace_name contains candidate substring
def contains_candidate(name, candidate):
    try:
        return candidate in name
    except Exception:
        return False

rows_list = []
for c in candidates:
    # anomaly_count_as_target: cmdb_id == candidate
    as_target = anom_rows[anom_rows['cmdb_id'] == c]
    anomaly_count_as_target = int(as_target.shape[0])
    earliest_anomaly_as_target = as_target['dt'].min() if not as_target.empty else pd.NaT

    # anomaly_count_involving_name: trace_name contains candidate substring
    involving = anom_rows[anom_rows['trace_name'].apply(lambda x: contains_candidate(x, c))]
    anomaly_count_involving_name = int(involving.shape[0])
    earliest_anomaly_involving_name = involving['dt'].min() if not involving.empty else pd.NaT

    # top_trace_names_involving: top up to 10 trace_name values and their anomaly counts
    if not involving.empty:
        top_counts = involving['trace_name'].value_counts().head(10)
        # represent as list of "name:count" strings for compactness
        top_list = [f"{name}::{int(cnt)}" for name, cnt in top_counts.items()]
    else:
        top_list = []

    rows_list.append({
        'candidate': c,
        'anomaly_count_as_target': anomaly_count_as_target,
        'anomaly_count_involving_name': anomaly_count_involving_name,
        'earliest_anomaly_as_target': earliest_anomaly_as_target,
        'earliest_anomaly_involving_name': earliest_anomaly_involving_name,
        'top_trace_names_involving': top_list
    })

candidate_anom_df = pd.DataFrame(rows_list)

# Global summaries
total_anomalous_trace_rows_for_candidates = int(candidate_anom_df['anomaly_count_as_target'].sum())

# For "involve any of the three": filter anomalous rows where trace_name contains any candidate substring
def trace_involves_any(name):
    return any(contains_candidate(name, c) for c in candidates)

involving_any = anom_rows[anom_rows['trace_name'].apply(lambda x: trace_involves_any(x))]

if not involving_any.empty:
    most_common_trace = involving_any['trace_name'].value_counts().idxmax()
    most_common_trace_count = int(involving_any['trace_name'].value_counts().max())
    most_common_trace_pair = (most_common_trace, most_common_trace_count)
else:
    most_common_trace_pair = (None, 0)

# Keep results compact: candidate_anom_df (3 rows) and two globals
candidate_anom_df, total_anomalous_trace_rows_for_candidates, most_common_trace_pair

```
Out[6]:
```
Summary (plain English, timestamps UTC+8):

Per candidate:
- cartservice2-0
  - anomalous trace rows where cmdb_id == cartservice2-0: 158
  - anomalous trace rows where trace_name contains "cartservice2-0": 121
  - earliest anomalous timestamp (as target): 2022-03-20 09:30:00+08:00
  - earliest anomalous timestamp (involving name): 2022-03-20 09:30:00+08:00
  - top trace(s) involving the name: trace.to_cartservice2-0.error_rate (58 occurrences) plus other to_cartservice2-0 traces.

- emailservice-0
  - anomalous trace rows where cmdb_id == emailservice-0: 113
  - anomalous trace rows where trace_name contains "emailservice-0": 113
  - earliest anomalous timestamp (as target): 2022-03-20 09:30:00+08:00
  - earliest anomalous timestamp (involving name): 2022-03-20 09:30:00+08:00
  - top trace(s) involving the name: trace.to_emailservice-0.error_rate (58 occurrences) and similar error_rate traces.

- shippingservice2-0
  - anomalous trace rows where cmdb_id == shippingservice2-0: 82
  - anomalous trace rows where trace_name contains "shippingservice2-0": 82
  - earliest anomalous timestamp (as target): 2022-03-20 09:30:00+08:00
  - earliest anomalous timestamp (involving name): 2022-03-20 09:30:00+08:00
  - top trace(s) involving the name: trace.to_shippingservice2-0.error_rate (58 occurrences) and related traces.

Global summaries:
- Total anomalous trace rows for the three candidates (sum of anomaly_count_as_target): 353
- Most common trace_name across anomalous rows that involve any of the three: "trace.to_emailservice-0.error_rate" (58 occurrences)

Interpretation: All three components show substantial trace anomalies beginning at 09:30+08, dominated by "to_<component>.error_rate" traces (each with ~58 occurrences), indicating elevated error rates directed at these services at the start of the incident window.

The original code execution output of IPython Kernel is also provided below for reference:

(            candidate  anomaly_count_as_target  anomaly_count_involving_name earliest_anomaly_as_target earliest_anomaly_involving_name                          top_trace_names_involving
0      cartservice2-0                      158                           121  2022-03-20 09:30:00+08:00       2022-03-20 09:30:00+08:00  [trace.to_cartservice2-0.error_rate::58, trace...
1      emailservice-0                      113                           113  2022-03-20 09:30:00+08:00       2022-03-20 09:30:00+08:00  [trace.to_emailservice-0.error_rate::58, trace...
2  shippingservice2-0                       82                            82  2022-03-20 09:30:00+08:00       2022-03-20 09:30:00+08:00  [trace.to_shippingservice2-0.error_rate::58, t..., 353, ('trace.to_emailservice-0.error_rate', 58))```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz
from collections import Counter

tz = pytz.timezone('Asia/Shanghai')

# Ensure trace_window_anom_df exists and anomalies are boolean
if 'trace_window_anom_df' not in globals():
    raise RuntimeError("trace_window_anom_df not found in kernel")
if 'is_anom' not in trace_window_anom_df.columns:
    trace_window_anom_df['is_anom'] = trace_window_anom_df.get('value', pd.Series()).notna()
trace_window_anom_df['is_anom'] = trace_window_anom_df['is_anom'].astype(bool)

anom_rows = trace_window_anom_df[trace_window_anom_df['is_anom']].copy()
anom_rows['trace_name'] = anom_rows['trace_name'].astype(str).fillna('')

candidates = ['cartservice2-0','emailservice-0','shippingservice2-0']

rows = []
for c in candidates:
    prefix_to = f"trace.to_{c}"
    prefix_from = f"trace.from_{c}"
    to_rows = anom_rows[anom_rows['trace_name'].str.startswith(prefix_to, na=False)].copy()
    from_rows = anom_rows[anom_rows['trace_name'].str.startswith(prefix_from, na=False)].copy()
    count_to_traces = int(to_rows.shape[0])
    count_from_traces = int(from_rows.shape[0])
    earliest_to_trace = to_rows['dt'].min() if not to_rows.empty else pd.NaT
    earliest_from_trace = from_rows['dt'].min() if not from_rows.empty else pd.NaT

    # Identify callers by matching anomalous 'from_' rows at same timestamps
    caller_counter = Counter()
    potential_callers = anom_rows[anom_rows['trace_name'].str.startswith('trace.from_', na=False)].copy()
    if not to_rows.empty and not potential_callers.empty:
        # extract caller name from 'trace.from_<caller>...'
        potential_callers['caller'] = potential_callers['trace_name'].str.replace(r'^trace\.from_', '', regex=True).str.split('.', n=1).str[0]
        for t in to_rows['dt'].unique():
            callers_at_t = potential_callers[potential_callers['dt'] == t]
            if callers_at_t.empty:
                continue
            caller_counter.update(callers_at_t['caller'].dropna().tolist())
    top_callers = [f"{caller}::{cnt}" for caller, cnt in caller_counter.most_common(5)]

    # max_metric_severity from per_component_df or fallback to true_candidate_df
    max_metric_severity = np.nan
    if 'per_component_df' in globals() and 'max_severity' in per_component_df.columns:
        tmp = per_component_df[per_component_df['cmdb_id'] == c]
        if not tmp.empty:
            max_metric_severity = float(tmp['max_severity'].iloc[0])
    elif 'true_candidate_df' in globals() and 'severity' in true_candidate_df.columns:
        tmp2 = true_candidate_df[true_candidate_df['cmdb_id'] == c]
        if not tmp2.empty:
            max_metric_severity = float(tmp2['severity'].max())

    rows.append({
        'candidate': c,
        'count_to_traces': count_to_traces,
        'count_from_traces': count_from_traces,
        'earliest_to_trace': earliest_to_trace,
        'earliest_from_trace': earliest_from_trace,
        'top_callers': top_callers,
        'max_metric_severity': max_metric_severity
    })

candidate_trace_role_df = pd.DataFrame(rows)

# Global summaries
if not candidate_trace_role_df.empty:
    top_row = candidate_trace_role_df.sort_values('count_to_traces', ascending=False).iloc[0]
    overall_most_to_traces_candidate = (top_row['candidate'], int(top_row['count_to_traces']))
else:
    overall_most_to_traces_candidate = (None, 0)

prefixes = [f"trace.to_{c}" for c in candidates] + [f"trace.from_{c}" for c in candidates]
mask_tofrom = anom_rows['trace_name'].str.startswith(tuple(prefixes), na=False)
relevant_rows = anom_rows[mask_tofrom]
combined_earliest_anomaly = relevant_rows['dt'].min() if not relevant_rows.empty else pd.NaT

# Return compact results
candidate_trace_role_df, overall_most_to_traces_candidate, combined_earliest_anomaly

```
Out[7]:
```
Summary (plain English, timestamps UTC+8):

Per candidate:
- cartservice2-0
  - anomalous "to" traces: 121
  - anomalous "from" traces: 0
  - earliest anomalous "to" trace: 2022-03-20 09:30:00+08
  - top callers (from anomalous rows): primarily checkoutservice-0 (≈923), checkoutservice-1 (≈90), … (other checkoutservice frontend callers)
  - max metric severity (per metrics): ~1.1e11 (very large)

- emailservice-0
  - anomalous "to" traces: 113
  - anomalous "from" traces: 0
  - earliest anomalous "to" trace: 2022-03-20 09:30:00+08
  - top callers: primarily checkoutservice-0 (≈890), checkoutservice-1 (≈87), …
  - max metric severity: ~1.79e12 (very large)

- shippingservice2-0
  - anomalous "to" traces: 82
  - anomalous "from" traces: 0
  - earliest anomalous "to" trace: 2022-03-20 09:30:00+08
  - top callers: primarily checkoutservice-0 (≈923), checkoutservice-1 (≈90), …
  - max metric severity: ~8.47

Global values:
- Candidate with the most "to" traces: cartservice2-0 (121)
- Combined earliest anomaly among all 'to_'/'from_' anomalous traces for these three: 2022-03-20 09:30:00+08

Interpretation (brief): All three services are primarily observed as targets (many "to_<service>" anomalies) beginning at 09:30+08, with checkoutservice pods appearing as the main callers. Metric severities for cartservice2-0 and emailservice-0 are extremely large and warrant immediate investigation (container memory failures, CPU throttling, I/O issues previously observed). Shippingservice2-0 shows lower numeric severity but still notable trace targeting.

The original code execution output of IPython Kernel is also provided below for reference:

(            candidate  count_to_traces  count_from_traces         earliest_to_trace earliest_from_trace                                        top_callers  max_metric_severity
0      cartservice2-0              121                  0 2022-03-20 09:30:00+08:00                 NaT  [checkoutservice-0::923, checkoutservice-1::90...         1.100000e+11
1      emailservice-0              113                  0 2022-03-20 09:30:00+08:00                 NaT  [checkoutservice-0::890, checkoutservice-1::87...         1.791000e+12
2  shippingservice2-0               82                  0 2022-03-20 09:30:00+08:00                 NaT  [checkoutservice-0::923, checkoutservice-1::90...         8.467712e+00, ('cartservice2-0', 121), Timestamp('2022-03-20 09:30:00+0800', tz='Asia/Shanghai'))```
```